In [1]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.id_to_char = {}
        self.char_to_id = {}

    def train(self, texts, N):
        """
        Training process of the BPE (Byte Pair Encoding) algorithm.
        :param texts: Training corpus (string)
        :param N: Target vocabulary size
        :return:
        """
        # Initial character-level tokenization
        unique_chrs = sorted(list(set(list(texts))))

        # Number of merge operations
        merge_times = N - len(unique_chrs)

        merge = {}

        # Initialize vocabulary
        id_to_char = {idx: char for idx, char in enumerate(unique_chrs)}
        char_to_id = {char: idx for idx, char in enumerate(unique_chrs)}

        ids = [char_to_id[c] for c in texts]
        print(ids)
        vac_size = len(unique_chrs) - 1

        for i in range(merge_times):
            if len(ids) == 1:
                break
            # Count frequencies of adjacent symbol pairs
            stats = self.stats(ids)
            # Find the most frequent adjacent pair
            pair = max(stats, key=stats.get)
            vac_size += 1
            # Merge the most frequent pair
            ids = self.merge_ids(ids, pair, vac_size)
            # Update vocabulary and merge dictionary
            id_to_char[vac_size] = id_to_char[pair[0]] + id_to_char[pair[1]]
            char_to_id[id_to_char[pair[0]] + id_to_char[pair[1]]] = vac_size
            merge[pair] = vac_size

        self.merges = merge
        self.char_to_id = char_to_id
        self.id_to_char = id_to_char

    def stats(self, ids):
        """
        Count the frequency of adjacent subword pairs.
        :param ids: List of token IDs
        :return: Dict of pair -> frequency
        """
        counts = {}
        for item in zip(ids[:-1], ids[1:]):
            counts[item] = counts.get(item, 0) + 1
        return counts

    def merge_ids(self, ids, pair, idx):
        """
        Update the list of token IDs after merging a given pair.
        :param ids: Current list of token IDs
        :param pair: The subword pair to be merged
        :param idx: The new vocabulary index for the merged pair
        :return: Updated list of token IDs
        """
        new_ids = []
        i = 0
        while i < len(ids):
            if ids[i] == pair[0] and i < len(ids) - 1 and ids[i + 1] == pair[1]:
                new_ids.append(idx)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        return new_ids

    def encode(self, text):
        """
        Tokenize the given text into IDs.
        :param text: Input text string
        :return: List of token IDs
        """
        ids = [self.char_to_id[c] for c in text]
        print(ids)
        # Iteratively merge until no more applicable pairs
        while len(ids) >= 2:
            stats = self.stats(ids)  # e.g., {(1,2):3, (3,4):5}
            print(f"stats: {stats}")
            pair = min(stats, key=lambda p: self.merges.get(p, float('inf')))
            print(f"pair: {pair}")
            if pair not in self.merges:
                break
            ids = self.merge_ids(ids, pair, self.merges[pair])
            print(f"ids: {ids}")
        return ids

    def decode(self, ids):
        """
        Decode a list of token IDs back into text.
        :param ids: List of token IDs
        :return: Decoded text string
        """
        return "".join([self.id_to_char[idx] for idx in ids])


In [2]:
t1 = BPETokenizer()

In [3]:
train_text = """
    hello, this is a training text. The tokenizer will split the text into words and assign an id
    to each word. This is a fantastic world.
    """
t1.train(texts=train_text, N=64)

[0, 1, 1, 1, 1, 11, 8, 14, 14, 16, 2, 1, 20, 11, 12, 19, 1, 12, 19, 1, 5, 1, 20, 18, 5, 12, 15, 12, 15, 10, 1, 20, 8, 22, 20, 3, 1, 4, 11, 8, 1, 20, 16, 13, 8, 15, 12, 23, 8, 18, 1, 21, 12, 14, 14, 1, 19, 17, 14, 12, 20, 1, 20, 11, 8, 1, 20, 8, 22, 20, 1, 12, 15, 20, 16, 1, 21, 16, 18, 7, 19, 1, 5, 15, 7, 1, 5, 19, 19, 12, 10, 15, 1, 5, 15, 1, 12, 7, 0, 1, 1, 1, 1, 20, 16, 1, 8, 5, 6, 11, 1, 21, 16, 18, 7, 3, 1, 4, 11, 12, 19, 1, 12, 19, 1, 5, 1, 9, 5, 15, 20, 5, 19, 20, 12, 6, 1, 21, 16, 18, 14, 7, 3, 0, 1, 1, 1, 1]


In [4]:
t1.encode("hello world")

[11, 8, 14, 14, 16, 1, 21, 16, 18, 14, 7]
stats: {(11, 8): 1, (8, 14): 1, (14, 14): 1, (14, 16): 1, (16, 1): 1, (1, 21): 1, (21, 16): 1, (16, 18): 1, (18, 14): 1, (14, 7): 1}
pair: (1, 21)
ids: [11, 8, 14, 14, 16, 28, 16, 18, 14, 7]
stats: {(11, 8): 1, (8, 14): 1, (14, 14): 1, (14, 16): 1, (16, 28): 1, (28, 16): 1, (16, 18): 1, (18, 14): 1, (14, 7): 1}
pair: (11, 8)
ids: [31, 14, 14, 16, 28, 16, 18, 14, 7]
stats: {(31, 14): 1, (14, 14): 1, (14, 16): 1, (16, 28): 1, (28, 16): 1, (16, 18): 1, (18, 14): 1, (14, 7): 1}
pair: (28, 16)
ids: [31, 14, 14, 16, 33, 18, 14, 7]
stats: {(31, 14): 1, (14, 14): 1, (14, 16): 1, (16, 33): 1, (33, 18): 1, (18, 14): 1, (14, 7): 1}
pair: (33, 18)
ids: [31, 14, 14, 16, 34, 14, 7]
stats: {(31, 14): 1, (14, 14): 1, (14, 16): 1, (16, 34): 1, (34, 14): 1, (14, 7): 1}
pair: (14, 14)
ids: [31, 36, 16, 34, 14, 7]
stats: {(31, 36): 1, (36, 16): 1, (16, 34): 1, (34, 14): 1, (14, 7): 1}
pair: (31, 36)


[31, 36, 16, 34, 14, 7]

In [5]:
t1.decode([31, 36, 16, 34, 14, 7])

'hello world'

In [6]:
t1.id_to_char

{0: '\n',
 1: ' ',
 2: ',',
 3: '.',
 4: 'T',
 5: 'a',
 6: 'c',
 7: 'd',
 8: 'e',
 9: 'f',
 10: 'g',
 11: 'h',
 12: 'i',
 13: 'k',
 14: 'l',
 15: 'n',
 16: 'o',
 17: 'p',
 18: 'r',
 19: 's',
 20: 't',
 21: 'w',
 22: 'x',
 23: 'z',
 24: '  ',
 25: ' t',
 26: 's ',
 27: 'is ',
 28: ' w',
 29: '\n  ',
 30: '\n    ',
 31: 'he',
 32: 'in',
 33: ' wo',
 34: ' wor',
 35: 'an',
 36: 'll',
 37: 'his ',
 38: 'his is ',
 39: 'his is a',
 40: ' te',
 41: ' tex',
 42: ' text',
 43: '. ',
 44: '. T',
 45: 'to',
 46: ' word',
 47: 'as',
 48: '\n    he',
 49: '\n    hell',
 50: '\n    hello',
 51: '\n    hello,',
 52: '\n    hello, t',
 53: '\n    hello, this is a',
 54: '\n    hello, this is a t',
 55: '\n    hello, this is a tr',
 56: '\n    hello, this is a tra',
 57: '\n    hello, this is a train',
 58: '\n    hello, this is a trainin',
 59: '\n    hello, this is a training',
 60: '\n    hello, this is a training text',
 61: '\n    hello, this is a training text. T',
 62: '\n    hello, this is a t